# CellMatch x Segmentation — GPU notebook → `submission.csv`

**Run all cells top to bottom.**

### On NYU Cloud Bursting (HPC)
1. Start **JupyterLab** on a **GPU** partition (`g2-standard-12` L4, or `c12m85-a100-1` A100), account `cs_gy_6923-2026fa`.
2. Open `/scratch/ts5789/cellmatch/CellMatch_Colab.ipynb` (or `CellMatch_HPC.ipynb` for the thin runner).
3. Kernel → **Python (cellmatch)** if that kernel exists after the setup cell.
4. Run All. Output: `/scratch/ts5789/cellmatch/submission.csv`.

### On Google Colab
1. Runtime → GPU (T4/A100/L4).
2. Checkpoints are stored in the Colab runtime by default; no Drive permission is required.
3. Data: Colab secrets `KAGGLE_USERNAME`/`KAGGLE_KEY`, or upload `testingpj-2.zip`.



In [ ]:
import os, sys
from pathlib import Path

ON_COLAB = False
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except Exception:
    ON_COLAB = False

if ON_COLAB:
    WORK = Path("/content/work")
    MODELS = WORK / "models"
    WORK.mkdir(exist_ok=True)
    MODELS.mkdir(exist_ok=True)
    os.chdir(WORK)
    get_ipython().system("pip -q install tifffile 'cellpose>=4.2'")
else:
    # NYU Burst / any cluster checkout
    candidates = [
        Path("/scratch/ts5789/cellmatch"),
        Path.cwd(),
        Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd(),
    ]
    WORK = next((p for p in candidates if (p / "pipeline.py").exists() or (p / "Project_2_Dataset").exists()), Path("/scratch/ts5789/cellmatch"))
    WORK = WORK.resolve()
    MODELS = WORK / "models"
    MODELS.mkdir(exist_ok=True)
    os.chdir(WORK)
    os.environ["SLURM_SUBMIT_DIR"] = str(WORK)
    os.environ["CELLPOSE_LOCAL_MODELS_PATH"] = str(WORK / "cellpose_weights")
    os.environ["PIP_CACHE_DIR"] = str(WORK / ".pip_cache")
    (WORK / "cellpose_weights").mkdir(exist_ok=True)
    env_py = WORK / "env" / "bin" / "python"
    if env_py.exists():
        # Ensure subsequent !commands can use project env if user selected system kernel
        os.environ["PATH"] = f"{env_py.parent}:{os.environ.get('PATH','')}"
    get_ipython().system("pip -q install tifffile 'cellpose>=4.2' scikit-image scipy pandas opencv-python-headless")

import torch
print("WORK =", WORK)
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), "Need a GPU runtime/session (Colab: Runtime→GPU; Burst: Jupyter on g2-standard-12 or A100 partition)"
get_ipython().system("nvidia-smi --query-gpu=name,memory.total --format=csv")


In [ ]:
import glob, zipfile, os
from pathlib import Path

DATA = WORK / "Project_2_Dataset"

def find_archive():
    hits = glob.glob(str(WORK / "**/testingpj-2*.zip"), recursive=True)
    if ON_COLAB:
        hits += glob.glob("/content/**/testingpj-2*.zip", recursive=True)
    return next((hit for hit in hits if zipfile.is_zipfile(hit)), None)

if not (DATA / "training/train_ground_truth.csv").exists():
    archive = find_archive()
    if archive is None and ON_COLAB:
        try:
            from google.colab import userdata
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
            get_ipython().system("kaggle competitions download -c testingpj-2 -p /content/work")
        except Exception as error:
            print("Kaggle secrets not available:", error)
        archive = find_archive()
    if archive is None and ON_COLAB:
        from google.colab import files
        print("Upload testingpj-2.zip")
        archive = str(WORK / next(iter(files.upload())))
    if archive is None:
        raise FileNotFoundError(f"No dataset at {DATA} and no testingpj-2.zip found under {WORK}")
    with zipfile.ZipFile(archive) as z:
        z.extractall(WORK)

print("training subjects:", sorted(p.name for p in (DATA / "training").glob("subject_*")))
print("test regions:", len(list((DATA / "hidden_test").glob("*/*"))))


## Pipeline code
Same modules as the local repo, written to `/content/work`.

In [ ]:
%%writefile cellmatch.py
"""Cell instance segmentation, registration, and Kaggle submission utilities.

Images and run lengths retain their original (row, column) coordinates throughout.
The routines here are intentionally independent of Kaggle's hidden labels.
"""

from __future__ import annotations

import csv
import json
from pathlib import Path

import cv2
import numpy as np
import tifffile
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
from skimage.segmentation import watershed


ROOT = Path(__file__).resolve().parent / "Project_2_Dataset"


def read_image(path: Path) -> np.ndarray:
    image = tifffile.imread(path)
    if image.ndim != 2:
        raise ValueError(f"Expected a 2-D image: {path}, shape={image.shape}")
    return image


def decode_rle(rle: str, shape: tuple[int, int]) -> np.ndarray:
    flat = np.zeros(shape[0] * shape[1], dtype=bool)
    runs = np.fromstring(rle, sep=" ", dtype=np.int64)
    if len(runs) % 2:
        raise ValueError("Odd number of RLE entries")
    for start, length in runs.reshape(-1, 2):
        if start < 0 or length <= 0 or start + length > flat.size:
            raise ValueError("Invalid RLE run")
        flat[start : start + length] = True
    return flat.reshape(shape)


def rle_to_labels(instances: dict[str, str], shape: tuple[int, int]) -> tuple[np.ndarray, list[str]]:
    labels = np.zeros(shape[0] * shape[1], np.int32)
    ids = list(instances)
    for index, cell_id in enumerate(ids, 1):
        runs = np.fromstring(instances[cell_id], sep=" ", dtype=np.int64)
        if len(runs) % 2:
            raise ValueError(f"Invalid RLE for {cell_id}")
        for start, length in runs.reshape(-1, 2):
            labels[start : start + length] = index
    return labels.reshape(shape), ids


def encode_rle(mask: np.ndarray) -> str:
    flat = np.asarray(mask, dtype=bool).ravel(order="C")
    edges = np.flatnonzero(np.diff(np.r_[False, flat, False].astype(np.int8)))
    return " ".join(f"{start} {end-start}" for start, end in edges.reshape(-1, 2))


def labels_to_rles(labels: np.ndarray, prefix: str) -> dict[str, str]:
    flat = np.asarray(labels, dtype=np.int32).ravel(order="C")
    edges = np.r_[0, np.flatnonzero(np.diff(flat)) + 1, len(flat)]
    runs: dict[int, list[str]] = {}
    for start, end in zip(edges[:-1], edges[1:]):
        cell = int(flat[start])
        if cell:
            runs.setdefault(cell, []).append(f"{start} {end-start}")
    return {f"{prefix}_{cell:06d}": " ".join(chunks) for cell, chunks in sorted(runs.items())}


def normalize(image: np.ndarray, modality: str) -> np.ndarray:
    """Multi-scale contrast normalized to a fairly stable [0,1] range."""
    image = image.astype(np.float32)
    low, high = np.percentile(image, [1, 99.8])
    image = np.clip((image - low) / max(high - low, 1), 0, 1)
    # Local contrast makes bright somata comparable across uneven illumination.
    background = cv2.GaussianBlur(image, (0, 0), 12 if modality == "invivo" else 9)
    contrast = image - background
    scale = np.percentile(contrast[contrast > 0], 99) if np.any(contrast > 0) else 1
    contrast = np.clip(contrast / max(scale, 0.02), 0, 1)
    return np.stack([image, contrast], axis=0).astype(np.float32)


def classical_segment(image: np.ndarray, modality: str, threshold: float = 0.19) -> np.ndarray:
    """Fallback segmentation based on cell-sized local contrast and watershed."""
    features = normalize(image, modality)
    contrast = features[1]
    sigma = 1.1 if modality == "invivo" else 0.9
    smooth = ndi.gaussian_filter(contrast, sigma)
    radius = 4 if modality == "invivo" else 3
    coords = peak_local_max(smooth, min_distance=radius, threshold_abs=threshold)
    markers = np.zeros(image.shape, np.int32)
    for i, (row, col) in enumerate(coords, 1):
        markers[row, col] = i
    foreground = smooth > (threshold * (0.48 if modality == "invivo" else 0.53))
    labels = watershed(-smooth, markers, mask=foreground)
    min_area, max_area = (18, 340) if modality == "invivo" else (15, 360)
    counts = np.bincount(labels.ravel())
    keep = (counts >= min_area) & (counts <= max_area)
    keep[0] = False
    labels[~keep[labels]] = 0
    _, inverse = np.unique(labels, return_inverse=True)
    return inverse.reshape(labels.shape).astype(np.int32)


def pq_score(pred: np.ndarray, truth: np.ndarray, cutoff: float = 0.75) -> tuple[float, int, int, int]:
    """Competition-style panoptic quality for a pair of disjoint label maps."""
    npred, ngt = int(pred.max()), int(truth.max())
    if not npred and not ngt:
        return 1.0, 0, 0, 0
    if not npred or not ngt:
        return 0.0, 0, npred, ngt
    joint = np.bincount((pred.astype(np.int64) * (ngt + 1) + truth).ravel(),
                        minlength=(npred + 1) * (ngt + 1)).reshape(npred + 1, ngt + 1)
    pc = joint.sum(axis=1)
    gc = joint.sum(axis=0)
    rows, cols = np.nonzero(joint[1:, 1:])
    rows += 1
    cols += 1
    iou = joint[rows, cols] / (pc[rows] + gc[cols] - joint[rows, cols])
    hits = iou > cutoff
    tp = int(hits.sum())
    fp, fn = npred - tp, ngt - tp
    denominator = tp + 0.5 * (fp + fn)
    return float(iou[hits].sum() / denominator), tp, fp, fn


def region_centers(labels: np.ndarray) -> np.ndarray:
    n = int(labels.max())
    if n == 0:
        return np.empty((0, 2), np.float32)
    yy, xx = np.indices(labels.shape)
    count = np.bincount(labels.ravel(), minlength=n + 1)[1:]
    x = np.bincount(labels.ravel(), weights=xx.ravel(), minlength=n + 1)[1:]
    y = np.bincount(labels.ravel(), weights=yy.ravel(), minlength=n + 1)[1:]
    return np.stack([x / np.maximum(count, 1), y / np.maximum(count, 1)], axis=1).astype(np.float32)


def write_submission(rows: list[dict], path: Path, sample_csv: Path = ROOT / "sample_submission.csv") -> None:
    columns = ["sample_id", "invivo_instances", "exvivo_instances", "match_pairs"]
    with sample_csv.open(newline="") as stream:
        expected = [row["sample_id"] for row in csv.DictReader(stream)]
    if [row["sample_id"] for row in rows] != expected:
        raise ValueError("Submission sample IDs/order do not match the sample file")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            iv, ex, pairs = row["invivo_instances"], row["exvivo_instances"], row["match_pairs"]
            assert len({pair[0] for pair in pairs}) == len(pairs)
            assert len({pair[1] for pair in pairs}) == len(pairs)
            assert all(a in iv and b in ex for a, b in pairs)
            writer.writerow({"sample_id": row["sample_id"], "invivo_instances": json.dumps(iv),
                             "exvivo_instances": json.dumps(ex), "match_pairs": json.dumps(pairs)})


In [ ]:
%%writefile learned.py
"""Small supervised U-Net for the two microscopy modalities."""

from __future__ import annotations

import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from scipy import ndimage as ndi
from skimage.segmentation import watershed

from cellmatch import ROOT, normalize, read_image, rle_to_labels


class ConvBlock(torch.nn.Module):
    def __init__(self, input_channels: int, output_channels: int):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Conv2d(input_channels, output_channels, 3, padding=1),
            torch.nn.GroupNorm(4, output_channels),
            torch.nn.LeakyReLU(0.1, inplace=True),
            torch.nn.Conv2d(output_channels, output_channels, 3, padding=1),
            torch.nn.GroupNorm(4, output_channels),
            torch.nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


class UNet(torch.nn.Module):
    def __init__(self, width: int = 16):
        super().__init__()
        self.down1 = ConvBlock(2, width)
        self.down2 = ConvBlock(width, width * 2)
        self.down3 = ConvBlock(width * 2, width * 4)
        self.down4 = ConvBlock(width * 4, width * 8)
        self.middle = ConvBlock(width * 8, width * 12)
        self.up4 = ConvBlock(width * 20, width * 8)
        self.up3 = ConvBlock(width * 12, width * 4)
        self.up2 = ConvBlock(width * 6, width * 2)
        self.up1 = ConvBlock(width * 3, width)
        self.output = torch.nn.Conv2d(width, 2, 1)
        self.pool = torch.nn.MaxPool2d(2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skips = []
        for block in (self.down1, self.down2, self.down3, self.down4):
            x = block(x)
            skips.append(x)
            x = self.pool(x)
        x = self.middle(x)
        for block, skip in zip((self.up4, self.up3, self.up2, self.up1), reversed(skips)):
            x = torch.nn.functional.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = block(torch.cat((x, skip), dim=1))
        return self.output(x)


def core_target(labels: np.ndarray) -> np.ndarray:
    """Erode instances separately, leaving a seed region for every cell."""
    positive = labels > 0
    result = positive.copy()
    for shift in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        result &= labels == np.roll(labels, shift, axis=(0, 1))
    result[:2] = False
    result[-2:] = False
    result[:, :2] = False
    result[:, -2:] = False
    return result


def resize_features(features: np.ndarray, shape: tuple[int, int]) -> np.ndarray:
    return np.stack([cv2.resize(channel, (shape[1], shape[0]), interpolation=cv2.INTER_LINEAR)
                     for channel in features.astype(np.float32)])


def load_training(modality: str, subjects: list[str] | None = None, scale: float = 1.0):
    frame = pd.read_csv(ROOT / "training/train_ground_truth.csv")
    records = []
    field = f"{modality}_instances"
    for row in frame.itertuples(index=False):
        subject = row.sample_id.split("__")[0]
        if subjects is not None and subject not in subjects:
            continue
        region = ROOT / "training" / row.sample_id.replace("__", "/")
        image = read_image(region / f"{modality}.tif")
        labels, _ = rle_to_labels(json.loads(getattr(row, field)), image.shape)
        features = normalize(image, modality)
        if scale != 1.0:
            # Small somata get more pixels; the network sees them at a friendlier size.
            shape = (round(image.shape[0] * scale), round(image.shape[1] * scale))
            features = resize_features(features, shape)
            labels = cv2.resize(labels, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
        features = features.astype(np.float16)
        core = core_target(labels)
        targets = np.stack((labels > 0, core), axis=0).astype(np.uint8)
        centers = ndi.center_of_mass(np.ones(labels.shape, np.uint8), labels, np.arange(1, labels.max() + 1))
        centers = np.asarray(centers, dtype=np.float32)
        records.append((row.sample_id, features, targets, centers))
    return records


class RandomPatches(torch.utils.data.Dataset):
    def __init__(self, records, patch: int = 128, length: int = 10000):
        self.records, self.patch, self.length = records, patch, length

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        _, features, targets, centers = self.records[np.random.randint(len(self.records))]
        height, width = features.shape[-2:]
        if len(centers) and np.random.random() < 0.7:
            cy, cx = centers[np.random.randint(len(centers))]
            cy += np.random.randint(-self.patch // 3, self.patch // 3 + 1)
            cx += np.random.randint(-self.patch // 3, self.patch // 3 + 1)
        else:
            cy, cx = np.random.randint(height), np.random.randint(width)
        y = int(np.clip(cy - self.patch // 2, 0, height - self.patch))
        x = int(np.clip(cx - self.patch // 2, 0, width - self.patch))
        image = features[:, y : y + self.patch, x : x + self.patch].astype(np.float32)
        target = targets[:, y : y + self.patch, x : x + self.patch].astype(np.float32)
        k = np.random.randint(4)
        image = np.rot90(image, k, axes=(1, 2))
        target = np.rot90(target, k, axes=(1, 2))
        if np.random.random() < 0.5:
            image = image[:, :, ::-1]
            target = target[:, :, ::-1]
        if np.random.random() < 0.5:
            image = image[:, ::-1]
            target = target[:, ::-1]
        gain = np.random.uniform(0.75, 1.35)
        gamma = np.random.uniform(0.75, 1.3)
        image = np.clip(image * gain, 0, 1) ** gamma
        if np.random.random() < 0.25:
            sigma = np.random.uniform(0.3, 0.8)
            image = np.stack([ndi.gaussian_filter(ch, sigma) for ch in image])
        image = np.clip(image + np.random.normal(0, np.random.uniform(0, 0.025), image.shape), 0, 1)
        return torch.from_numpy(np.ascontiguousarray(image, dtype=np.float32)), torch.from_numpy(
            np.ascontiguousarray(target, dtype=np.float32)
        )


def loss_function(logits: torch.Tensor, targets: torch.Tensor, modality: str) -> torch.Tensor:
    pos_weight = torch.tensor([3.0, 5.0] if modality == "invivo" else [6.0, 8.0], device=logits.device)
    bce = torch.nn.functional.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pos_weight[None, :, None, None]
    )
    probs = torch.sigmoid(logits)
    intersection = (probs * targets).sum((0, 2, 3))
    dice = 1 - (2 * intersection + 1) / (probs.sum((0, 2, 3)) + targets.sum((0, 2, 3)) + 1)
    return bce + dice.mean()


@torch.inference_mode()
def predict_probabilities(model: UNet, image: np.ndarray, modality: str, device: str = "cpu",
                          tile: int = 256, overlap: int = 32, tta: bool = False,
                          scale: float = 1.0) -> np.ndarray:
    features = normalize(image, modality)
    if scale != 1.0:
        features = resize_features(features, (round(image.shape[0] * scale),
                                              round(image.shape[1] * scale)))
    height, width = features.shape[1:]
    output = np.zeros((2, height, width), np.float32)
    weights = np.zeros((height, width), np.float32)
    stride = tile - overlap
    ys = list(range(0, max(height - tile, 0), stride)) + [max(height - tile, 0)]
    xs = list(range(0, max(width - tile, 0), stride)) + [max(width - tile, 0)]
    window = np.ones((tile, tile), np.float32)
    edge = np.linspace(0.15, 1, overlap, dtype=np.float32)
    window[:overlap] *= edge[:, None]
    window[-overlap:] *= edge[::-1, None]
    window[:, :overlap] *= edge[None, :]
    window[:, -overlap:] *= edge[None, ::-1]
    model.eval()
    for y in ys:
        for x in xs:
            crop = features[:, y : y + tile, x : x + tile]
            crop = np.pad(crop, ((0, 0), (0, tile - crop.shape[1]), (0, tile - crop.shape[2])), mode="reflect")
            tensor = torch.from_numpy(crop[None]).to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device == "cuda"):
                if tta:
                    batch = torch.cat((tensor, tensor.flip(-1), tensor.flip(-2), tensor.transpose(-1, -2)))
                    prediction = torch.sigmoid(model(batch).float())
                    prob = torch.stack((prediction[0], prediction[1].flip(-1),
                                        prediction[2].flip(-2), prediction[3].transpose(-1, -2))).mean(0).cpu().numpy()
                else:
                    prob = torch.sigmoid(model(tensor).float())[0].cpu().numpy()
            h, w = min(tile, height - y), min(tile, width - x)
            output[:, y : y + h, x : x + w] += prob[:, :h, :w] * window[:h, :w]
            weights[y : y + h, x : x + w] += window[:h, :w]
    output /= weights[None]
    if scale != 1.0:
        output = resize_features(output, image.shape)
    return output


def decode_instances(probs: np.ndarray, modality: str, body_threshold: float = 0.5,
                     core_threshold: float = 0.5) -> np.ndarray:
    """Marker watershed on the body/core maps, with the hard size gates."""
    body = probs[0] >= body_threshold
    core = (probs[1] >= core_threshold) & body
    core = ndi.binary_opening(core, iterations=1)
    markers, count = ndi.label(core)
    sizes = np.bincount(markers.ravel())
    markers[sizes[markers] < 4] = 0
    # Guarantee a marker for a confidently detected cell with no core peak.
    components, component_count = ndi.label(body)
    covered = np.bincount(components[markers > 0], minlength=component_count + 1)
    candidates = np.flatnonzero((covered == 0) & (np.bincount(components.ravel()) >= 15))
    for component in candidates[candidates > 0]:
        location = np.argmax(np.where(components == component, probs[1], -1))
        markers.flat[location] = count + component
    labels = watershed(-probs[1], markers, mask=body).astype(np.int32)
    counts = np.bincount(labels.ravel())
    low, high = (20, 330) if modality == "invivo" else (17, 350)
    keep = (counts >= low) & (counts <= high)
    keep[0] = False
    labels[~keep[labels]] = 0
    _, inverse = np.unique(labels, return_inverse=True)
    return inverse.reshape(labels.shape).astype(np.int32)


def instance_features(labels: np.ndarray, probs: np.ndarray | None = None) -> np.ndarray:
    """Per-instance (core mean, circularity, eccentricity, bbox fill), rows in label order.

    Without probabilities the core-mean column is zero.
    """
    from skimage.measure import regionprops

    features = np.zeros((int(labels.max()), 4), np.float32)
    for prop in regionprops(labels, intensity_image=None if probs is None else probs[1]):
        perimeter = max(prop.perimeter, 1.0)
        y0, x0, y1, x1 = prop.bbox
        features[prop.label - 1] = (0.0 if probs is None else prop.intensity_mean,
                                    4 * np.pi * prop.area / perimeter ** 2,
                                    prop.eccentricity, prop.area / max((y1 - y0) * (x1 - x0), 1))
    return features


def filter_instances(labels: np.ndarray, features: np.ndarray, min_core: float = 0.0,
                     min_circularity: float = 0.0, max_eccentricity: float = 1.0,
                     min_fill: float = 0.0) -> np.ndarray:
    """Drop non-compact blobs (bright axons/dendrites) and weak detections."""
    keep = np.r_[False, (features[:, 0] >= min_core) & (features[:, 1] >= min_circularity)
                 & (features[:, 2] <= max_eccentricity) & (features[:, 3] >= min_fill)]
    labels = np.where(keep[labels], labels, 0)
    _, inverse = np.unique(labels, return_inverse=True)
    return inverse.reshape(labels.shape).astype(np.int32)


def probabilities_to_labels(probs: np.ndarray, modality: str, body_threshold: float = 0.5,
                            core_threshold: float = 0.5, **filters) -> np.ndarray:
    labels = decode_instances(probs, modality, body_threshold, core_threshold)
    if not filters or not labels.max():
        return labels
    return filter_instances(labels, instance_features(labels, probs), **filters)


def load_model(path: Path, device: str = "cpu") -> UNet:
    state = torch.load(path, map_location=device, weights_only=True)
    model = UNet(width=state["down1.layers.0.weight"].shape[0])
    model.load_state_dict(state)
    return model.to(device).eval()




In [ ]:
%%writefile registration.py
"""Robust registration of sparse cell centers across imaging modalities."""

from __future__ import annotations

import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

from cellmatch import ROOT, read_image, region_centers, rle_to_labels


def transform(points: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    return points @ matrix[:, :2].T + matrix[:, 2]


def initial_candidates(invivo: np.ndarray, exvivo: np.ndarray, angle_limit: int = 45,
                       bin_size: float = 12, count: int = 36) -> list[np.ndarray]:
    """Hough voting for angle, scale, and translation from all point pairs."""
    if len(invivo) < 3 or len(exvivo) < 3:
        return []
    angles = np.arange(-angle_limit, angle_limit + 0.1, 3)
    scales = np.arange(0.79, 1.141, 0.04)
    pool = []
    # In-vivo coordinates are in a much smaller canvas; translations are bounded.
    limit = 2500
    bins = int(2 * limit / bin_size) + 1
    for angle in angles:
        theta = np.deg2rad(angle)
        base = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
        for scale in scales:
            linear = scale * base
            rotated = invivo @ linear.T
            differences = (exvivo[:, None, :] - rotated[None, :, :]).reshape(-1, 2)
            xy = np.floor((differences + limit) / bin_size).astype(np.int32)
            valid = (xy >= 0).all(axis=1) & (xy < bins).all(axis=1)
            votes = np.bincount(xy[valid, 1] * bins + xy[valid, 0], minlength=bins * bins)
            peak_ids = np.argpartition(votes, -3)[-3:]
            for peak in peak_ids:
                ix, iy = peak % bins, peak // bins
                around = (np.abs(xy[valid, 0] - ix) <= 1) & (np.abs(xy[valid, 1] - iy) <= 1)
                if around.sum() < 3:
                    continue
                shift = np.median(differences[valid][around], axis=0)
                matrix = np.c_[linear, shift]
                _, _, distances = paired_nearest(invivo, exvivo, matrix, 16)
                score = np.maximum(0, 1 - distances / 16).sum()
                pool.append((float(score), matrix))
    pool.sort(key=lambda pair: pair[0], reverse=True)
    return [matrix for _, matrix in pool[:count]]


def fixed_linear_candidates(invivo: np.ndarray, exvivo: np.ndarray, linear: np.ndarray,
                            bin_size: float = 8, count: int = 20) -> list[np.ndarray]:
    """Find translations when a subject-level linear transform is known approximately."""
    differences = (exvivo[:, None, :] - (invivo @ linear.T)[None, :, :]).reshape(-1, 2)
    limit = 2500
    bins = int(2 * limit / bin_size) + 1
    xy = np.floor((differences + limit) / bin_size).astype(np.int32)
    valid = (xy >= 0).all(axis=1) & (xy < bins).all(axis=1)
    histogram = np.bincount(xy[valid, 1] * bins + xy[valid, 0], minlength=bins * bins).reshape(bins, bins)
    histogram = gaussian_filter(histogram.astype(np.float32), 0.8)
    pool = []
    for _ in range(count):
        peak = np.unravel_index(np.argmax(histogram), histogram.shape)
        if histogram[peak] == 0:
            break
        nearby = (np.abs(xy[valid, 0] - peak[1]) <= 2) & (np.abs(xy[valid, 1] - peak[0]) <= 2)
        shift = np.median(differences[valid][nearby], axis=0)
        pool.append(np.c_[linear, shift])
        y, x = peak
        histogram[max(0, y - 3) : y + 4, max(0, x - 3) : x + 4] = 0
    return pool


def paired_nearest(invivo: np.ndarray, exvivo: np.ndarray, matrix: np.ndarray,
                   distance: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    projected = transform(invivo, matrix)
    tree = cKDTree(exvivo)
    nearest_distance, nearest_index = tree.query(projected, distance_upper_bound=distance)
    valid = np.isfinite(nearest_distance)
    sources = np.flatnonzero(valid)
    if not len(sources):
        return np.array([], int), np.array([], int), np.array([], float)
    destinations = nearest_index[valid]
    distances = nearest_distance[valid]
    # Retain a single source for each ex-vivo cell.
    order = np.argsort(distances)
    used = set()
    keep = []
    for k in order:
        if int(destinations[k]) not in used:
            used.add(int(destinations[k]))
            keep.append(k)
    return sources[keep], destinations[keep], distances[keep]


def refine(invivo: np.ndarray, exvivo: np.ndarray, initial: np.ndarray,
           target_weights: np.ndarray | None = None) -> tuple[np.ndarray, float]:
    matrix = initial.copy()
    for distance in (30, 20, 12, 7):
        sources, destinations, _ = paired_nearest(invivo, exvivo, matrix, distance)
        if len(sources) < 3:
            break
        fit, inliers = cv2.estimateAffine2D(invivo[sources], exvivo[destinations],
                                             method=cv2.RANSAC, ransacReprojThreshold=max(4, distance * 0.5),
                                             maxIters=2000, refineIters=20)
        if fit is None or not 0.65 < abs(np.linalg.det(fit[:, :2])) < 1.3:
            break
        matrix = fit
    _, destination, distances = paired_nearest(invivo, exvivo, matrix, 8)
    # True registrations support many close, unique cell pairs.
    pair_weights = np.ones(len(distances)) if target_weights is None else target_weights[destination]
    score = np.sum(pair_weights * (1 + np.maximum(0, 1 - distances / 8)))
    return matrix, float(score)


def register_points(invivo: np.ndarray, exvivo: np.ndarray) -> tuple[np.ndarray | None, float]:
    candidates = initial_candidates(invivo, exvivo)
    if not candidates:
        return None, 0
    results = [refine(invivo, exvivo, candidate) for candidate in candidates]
    return max(results, key=lambda result: result[1])


# Median in-vivo to ex-vivo linear transforms estimated from the 47 labeled
# regions, using verified pairs only.  Translation is estimated per region.
TRAINING_LINEAR_PRIORS = (
    np.array([[0.967, -0.161], [0.121, 0.909]], dtype=np.float32),
    np.array([[0.982, 0.227], [-0.256, 0.927]], dtype=np.float32),
)


def estimate_linear_priors(exclude_subjects: set[str] | None = None) -> tuple[np.ndarray, ...]:
    """Fit orientation-clustered linear priors from verified GT pairs.

    When exclude_subjects is set, those mice are held out (leak-free CV).
    """
    frame = pd.read_csv(ROOT / "training/train_ground_truth.csv")
    matrices = []
    for row in frame.itertuples(index=False):
        subject = row.sample_id.split("__")[0]
        if exclude_subjects and subject in exclude_subjects:
            continue
        path = ROOT / "training" / row.sample_id.replace("__", "/")
        iv, iv_ids = rle_to_labels(json.loads(row.invivo_instances),
                                   read_image(path / "invivo.tif").shape)
        ex, ex_ids = rle_to_labels(json.loads(row.exvivo_instances),
                                   read_image(path / "exvivo.tif").shape)
        iv_centers, ex_centers = region_centers(iv), region_centers(ex)
        iv_lookup = {cell_id: i for i, cell_id in enumerate(iv_ids)}
        ex_lookup = {cell_id: i for i, cell_id in enumerate(ex_ids)}
        pairs = [(iv_lookup[a], ex_lookup[b]) for a, b in json.loads(row.match_pairs)
                 if a in iv_lookup and b in ex_lookup]
        if len(pairs) < 3:
            continue
        fit, _ = cv2.estimateAffine2D(
            iv_centers[[i for i, _ in pairs]].astype(np.float32),
            ex_centers[[j for _, j in pairs]].astype(np.float32),
            method=cv2.RANSAC, ransacReprojThreshold=5)
        if fit is None:
            continue
        linear = fit[:, :2]
        if not 0.65 < abs(np.linalg.det(linear)) < 1.3:
            continue
        matrices.append(linear.astype(np.float32))
    # Two orientation clusters, matching the order of TRAINING_LINEAR_PRIORS.
    positive = [m for m in matrices if np.arctan2(m[1, 0], m[0, 0]) >= 0]
    negative = [m for m in matrices if np.arctan2(m[1, 0], m[0, 0]) < 0]
    if not positive or not negative:
        return TRAINING_LINEAR_PRIORS
    return (
        np.median(np.stack(positive), axis=0).astype(np.float32),
        np.median(np.stack(negative), axis=0).astype(np.float32),
    )


def register_with_priors(invivo: np.ndarray, exvivo: np.ndarray,
                         mode: int | None = None,
                         target_weights: np.ndarray | None = None,
                         return_candidates: bool = False,
                         priors: tuple[np.ndarray, ...] | None = None,
                         ) -> tuple[np.ndarray | None, float, int] | list[tuple[np.ndarray, float, int]]:
    """Search label-derived geometric priors and solve translation from data."""
    if len(invivo) < 3 or len(exvivo) < 3:
        return None, 0, -1
    prior_set = priors if priors is not None else TRAINING_LINEAR_PRIORS
    results = []
    modes = range(len(prior_set)) if mode is None else (mode,)
    for mode_index in modes:
        base = prior_set[mode_index]
        for angle in (-5, 0, 5):
            theta = np.deg2rad(angle)
            rotation = np.array([[np.cos(theta), -np.sin(theta)],
                                 [np.sin(theta), np.cos(theta)]])
            for scale in (0.95, 1.0, 1.05):
                linear = (scale * rotation @ base).astype(np.float32)
                for candidate in fixed_linear_candidates(invivo, exvivo, linear, count=4):
                    matrix, score = refine(invivo, exvivo, candidate, target_weights)
                    results.append((matrix, score, mode_index))
    if return_candidates:
        return sorted(results, key=lambda item: item[1], reverse=True)
    return max(results, key=lambda item: item[1]) if results else (None, 0, -1)


def blob_map(points: np.ndarray, shape: tuple[int, int], sigma: float) -> np.ndarray:
    image = np.zeros(shape, np.float32)
    ij = np.round(points[:, ::-1]).astype(int)
    inside = (ij >= 0).all(axis=1) & (ij[:, 0] < shape[0]) & (ij[:, 1] < shape[1])
    np.add.at(image, (ij[inside, 0], ij[inside, 1]), 1.0)
    return cv2.GaussianBlur(image, (0, 0), sigma)


def register_ncc(invivo: np.ndarray, exvivo: np.ndarray, priors: tuple[np.ndarray, ...],
                 down: float = 2.0, sigma: float = 1.2, top: int = 3) -> tuple[np.ndarray | None, float, int]:
    """Normalized cross-correlation of cell-density maps over every translation.

    Normalization discounts dense clusters of false ex-vivo detections, which is what
    fools the raw pair-count score used by register_with_priors.
    """
    if len(invivo) < 3 or len(exvivo) < 3:
        return None, 0, -1
    target = exvivo / down
    candidates = []
    for mode, base in enumerate(priors):
        for angle in np.arange(-9, 9.1, 1.5):
            theta = np.deg2rad(angle)
            rotation = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
            for scale in np.arange(0.93, 1.101, 0.02):
                linear = scale * rotation @ base
                projected = invivo @ linear.T / down
                origin = projected.min(axis=0) - 4
                template_points = projected - origin
                template = blob_map(template_points, (int(template_points[:, 1].max()) + 5,
                                                      int(template_points[:, 0].max()) + 5), sigma)
                pad = np.array(template.shape) // 2
                image = blob_map(target + pad[::-1], (int(target[:, 1].max()) + 5 + 2 * pad[0],
                                                      int(target[:, 0].max()) + 5 + 2 * pad[1]), sigma)
                image = cv2.copyMakeBorder(image, 0, max(0, template.shape[0] - image.shape[0]),
                                           0, max(0, template.shape[1] - image.shape[1]), cv2.BORDER_CONSTANT)
                _, value, _, location = cv2.minMaxLoc(cv2.matchTemplate(image, template, cv2.TM_CCORR_NORMED))
                shift = (np.array(location, float) - origin - pad[::-1]) * down
                candidates.append((value, np.c_[linear, shift], mode))
    candidates.sort(key=lambda item: item[0], reverse=True)
    refined = [(*refine(invivo, exvivo, matrix), mode) for _, matrix, mode in candidates[:top]]
    return max(refined, key=lambda item: item[1])


def match_points(invivo: np.ndarray, exvivo: np.ndarray, matrix: np.ndarray,
                 max_distance: float = 5.0, method: str = "greedy") -> list[tuple[int, int, float]]:
    if method == "greedy" or len(invivo) == 0 or len(exvivo) == 0:
        sources, destinations, distances = paired_nearest(invivo, exvivo, matrix, max_distance)
        return [(int(i), int(j), float(d)) for i, j, d in zip(sources, destinations, distances)]
    # Hungarian: the most one-to-one pairs within max_distance, then the least total distance.
    close = cKDTree(transform(invivo, matrix)).sparse_distance_matrix(
        cKDTree(exvivo), max_distance, output_type="ndarray")
    if not len(close):
        return []
    rows, row_index = np.unique(close["i"], return_inverse=True)
    cols, col_index = np.unique(close["j"], return_inverse=True)
    cost = np.full((len(rows), len(cols)), 1e6)
    cost[row_index, col_index] = close["v"]
    chosen_rows, chosen_cols = linear_sum_assignment(cost)
    return [(int(rows[i]), int(cols[j]), float(cost[i, j]))
            for i, j in zip(chosen_rows, chosen_cols) if cost[i, j] < 1e6]


In [ ]:
%%writefile pipeline.py
"""End-to-end pipeline used by the Colab notebook.

Stages: leave-one-mouse-out training, held-out prediction cache, leak-free search of
segmentation + matching settings on the competition score, final models, submission.

Segmentation sources compete in the search: our U-Net at each input scale ("unet_x1", "unet_x2"),
Cellpose-SAM fine-tuned on these masks at one or more training budgets ("cellpose",
"cellpose_long"), and Cellpose masks filtered by the U-Net's core probability and shape features
("hybrid").
"""

from __future__ import annotations

import itertools
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.spatial import cKDTree

from cellmatch import (ROOT, labels_to_rles, normalize, pq_score, read_image, region_centers,
                       rle_to_labels, write_submission)
from learned import (RandomPatches, UNet, decode_instances, filter_instances, instance_features,
                     load_model, load_training, loss_function, predict_probabilities)
from registration import estimate_linear_priors, match_points, register_ncc, register_with_priors, transform

SUBJECTS = ("subject_5d294c", "subject_b2ba5e", "subject_db6b8b")
MODALITIES = ("invivo", "exvivo")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODELS = ROOT.parent / "models"
WORKERS = int(os.environ.get("SLURM_CPUS_PER_TASK", "2"))  # DataLoader workers for U-Net training

BODIES = (0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75)
CORES = {"invivo": (0.4, 0.5, 0.6), "exvivo": (0.5,)}
FILTERS = {
    "invivo": [{}, {"min_core": 0.2}, {"min_core": 0.3},
               {"min_core": 0.2, "min_circularity": 0.4, "max_eccentricity": 0.97, "min_fill": 0.3}],
    "exvivo": [{}, {"min_core": 0.3}, {"min_core": 0.4},
               {"min_core": 0.3, "min_circularity": 0.45, "max_eccentricity": 0.95, "min_fill": 0.35},
               {"min_core": 0.3, "min_circularity": 0.55, "max_eccentricity": 0.93, "min_fill": 0.4},
               {"min_circularity": 0.5, "max_eccentricity": 0.94, "min_fill": 0.38}],
}
METHODS = ("greedy", "hungarian")
DISTANCES = (3, 4, 5, 6, 7, 8, 10, 12)
BRIGHTNESS_LEVELS = (0.0, 0.45, 0.55, 0.65, 0.75, 0.85)
RATIOS = (None, 1.5, 2.0, 2.5, 3.0)          # second-nearest / nearest distance required for a pair
CELLPOSE_GRID = tuple(itertools.product((-1.0, 0.0, 1.0), (0.1, 0.2, 0.3, 0.4)))  # (cellprob, flow)
FIELD_PRIOR = np.array([0.48, 0.54])  # mean true in-vivo field centre on the ex-vivo canvas, training mice
CELLPOSE_TILE = 128
REGISTRATIONS = ("none", "soma", "ncc")
CENTRE_GATES = (None, 0.06, 0.08, 0.10, 0.14)


def model_path(modality: str, width: int, scale: float, tag: str, seed: int) -> Path:
    return MODELS / f"{modality}_w{width}_x{scale:g}_{tag}_seed{seed}.pt"


def seed_worker(worker_id: int) -> None:
    np.random.seed((torch.initial_seed() + worker_id) % 2**32)


def train(modality: str, subjects, output: Path, steps: int, width: int, scale: float,
          seed: int, batch: int = 16) -> Path:
    if output.exists():
        print("reusing", output.name, flush=True)
        return output
    torch.manual_seed(seed)
    np.random.seed(seed)
    records = load_training(modality, list(subjects), scale)
    loader = torch.utils.data.DataLoader(
        RandomPatches(records, length=steps * batch), batch_size=batch, num_workers=WORKERS,
        worker_init_fn=seed_worker, pin_memory=DEVICE == "cuda")
    model = UNet(width).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=3e-5)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
    started = time.monotonic()
    model.train()
    for step, (images, masks) in enumerate(loader, 1):
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            logits = model(images)
        loss = loss_function(logits.float(), masks, modality)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        if step % 500 == 0 or step == steps:
            print(f"  {output.name} step {step}/{steps} loss={loss.item():.4f} "
                  f"{time.monotonic() - started:.0f}s", flush=True)
        if step >= steps:
            break
    output.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.cpu().state_dict(), output)
    return output


def unet_source(scale: float) -> str:
    return f"unet_x{scale:g}"


def cellpose_path(modality: str, tag: str) -> Path:
    return MODELS / f"{modality}_cpsam_{tag}"


def cellpose_model(path: Path | None = None):
    from cellpose import models

    return models.CellposeModel(gpu=DEVICE == "cuda", pretrained_model=str(path) if path else "cpsam_v2",
                                use_bfloat16=DEVICE == "cuda" and torch.cuda.is_bf16_supported())


def percentile_normalize(image: np.ndarray) -> np.ndarray:
    low, high = np.percentile(image, [1, 99])
    return ((image.astype(np.float32) - low) / max(high - low, 1e-6)).astype(np.float32)


def cellpose_tiles(modality: str, subjects, per_image: int = 40, seed: int = 0):
    """Crops centred on labelled cells, normalized with whole-image percentiles.

    Cellpose trains on random 256-px crops of the image upsampled to ~30-px cells; on a full
    ex-vivo canvas such a crop holds well under one labelled cell on average.
    """
    rng = np.random.default_rng(seed)
    frame = pd.read_csv(ROOT / "training/train_ground_truth.csv")
    images, labels = [], []
    size, half = CELLPOSE_TILE, CELLPOSE_TILE // 2
    for row in frame.itertuples(index=False):
        if row.sample_id.split("__")[0] not in subjects:
            continue
        image = percentile_normalize(read_image(region_path(row.sample_id) / f"{modality}.tif"))
        label, _ = rle_to_labels(json.loads(getattr(row, f"{modality}_instances")), image.shape)
        centres = region_centers(label)
        for x, y in centres[rng.choice(len(centres), min(per_image, len(centres)), replace=False)]:
            y0 = int(np.clip(round(y) - half, 0, image.shape[0] - size))
            x0 = int(np.clip(round(x) - half, 0, image.shape[1] - size))
            _, crop = np.unique(label[y0:y0 + size, x0:x0 + size], return_inverse=True)
            images.append(image[y0:y0 + size, x0:x0 + size])
            labels.append(crop.reshape(size, size).astype(np.int32))
    return images, labels


def train_cellpose(modality: str, subjects, output: Path, epochs: int, images_per_epoch: int,
                   batch: int = 4) -> Path:
    if output.exists():
        print("reusing", output.name, flush=True)
        return output
    from cellpose import train as cellpose_train

    started = time.monotonic()
    images, labels = cellpose_tiles(modality, list(subjects))
    model = cellpose_model()
    cellpose_train.train_seg(model.net, train_data=images, train_labels=labels, normalize=False,
                             rescale=True, batch_size=batch, n_epochs=epochs,
                             nimg_per_epoch=images_per_epoch, learning_rate=1e-5, weight_decay=0.1,
                             min_train_masks=1, save_path=str(MODELS.parent), model_name=output.name)
    print(f"  {output.name}: {len(images)} tiles, {time.monotonic() - started:.0f}s", flush=True)
    return output


def cellpose_flows(models, image: np.ndarray) -> tuple:
    """Flows and cell probability at the original resolution, averaged over one or more models."""
    models = models if isinstance(models, (list, tuple)) else [models]
    flows, probabilities, diameters = [], [], []
    for model in models:
        diameter = float(model.net.diam_labels.item())
        _, output, _ = model.eval(image.astype(np.float32), diameter=diameter, compute_masks=False,
                                  batch_size=16)
        flows.append(output[1])
        probabilities.append(output[2])
        diameters.append(diameter)
    return (np.mean(flows, axis=0).astype(np.float16), np.mean(probabilities, axis=0).astype(np.float16),
            int(200 * np.mean(diameters) / 30))


def cellpose_labels(flows: tuple, modality: str, config: dict) -> np.ndarray:
    from cellpose import dynamics

    dP, cellprob, niter = flows
    masks = dynamics.resize_and_compute_masks(
        dP.astype(np.float32), cellprob.astype(np.float32), niter=niter,
        cellprob_threshold=config["cellprob"], flow_threshold=config["flow"], min_size=10,
        device=torch.device(DEVICE))
    counts = np.bincount(masks.ravel())
    low, high = (20, 330) if modality == "invivo" else (17, 350)
    keep = (counts >= low) & (counts <= high)
    keep[0] = False
    masks = np.where(keep[masks], masks, 0)
    _, inverse = np.unique(masks, return_inverse=True)
    return inverse.reshape(masks.shape).astype(np.int32)


def fold_cellpose_flows(truth: dict, modality: str, source: str = "cellpose") -> dict:
    """Cellpose flows for every training region from the fine-tune that never saw its mouse."""
    cache = {}
    for subject in SUBJECTS:
        model = cellpose_model(cellpose_path(modality, f"fold_{subject}{source.removeprefix('cellpose')}"))
        for sample_id in (s for s, t in truth.items() if t["subject"] == subject):
            cache[sample_id] = cellpose_flows(model, read_image(region_path(sample_id) / f"{modality}.tif"))
    return cache


def region_path(sample_id: str, split: str = "training") -> Path:
    return ROOT / split / sample_id.replace("__", "/")


def load_truth() -> dict:
    frame = pd.read_csv(ROOT / "training/train_ground_truth.csv")
    truth = {}
    for row in frame.itertuples(index=False):
        item = {"subject": row.sample_id.split("__")[0],
                "pairs": {tuple(pair) for pair in json.loads(row.match_pairs)}}
        for modality in MODALITIES:
            shape = read_image(region_path(row.sample_id) / f"{modality}.tif").shape
            item[modality] = rle_to_labels(json.loads(getattr(row, f"{modality}_instances")), shape)
        truth[row.sample_id] = item
    return truth


def fold_probabilities(truth: dict, modality: str, width: int, scale: float, tta: bool = True) -> dict:
    """Probabilities for every training region from the model that never saw its mouse."""
    cache = {}
    for subject in SUBJECTS:
        model = load_model(model_path(modality, width, scale, f"fold_{subject}", 0), DEVICE)
        for sample_id in (s for s, t in truth.items() if t["subject"] == subject):
            image = read_image(region_path(sample_id) / f"{modality}.tif")
            cache[sample_id] = predict_probabilities(model, image, modality, DEVICE, tta=tta,
                                                     scale=scale).astype(np.float16)
    return cache


def decode(probs, modality: str, config: dict) -> np.ndarray:
    if config["source"].startswith("cellpose"):
        return cellpose_labels(probs, modality, config)
    if config["source"] == "hybrid":
        flows, unet_probs = probs
        labels = cellpose_labels(flows, modality, config)
        if not labels.max():
            return labels
        return filter_instances(labels, instance_features(labels, unet_probs.astype(np.float32)),
                                **FILTERS[modality][config["filter"]])
    probs = probs.astype(np.float32)
    labels = decode_instances(probs, modality, config["body"], config["core"])
    if not labels.max():
        return labels
    return filter_instances(labels, instance_features(labels, probs), **FILTERS[modality][config["filter"]])


def pq_table(probs: dict, truth: dict, modality: str, scale: float) -> list[dict]:
    rows = []
    filters = FILTERS[modality]
    for body, core in itertools.product(BODIES, CORES[modality]):
        scores = np.zeros((len(filters), len(probs)))
        for k, (sample_id, region_probs) in enumerate(probs.items()):
            region_probs = region_probs.astype(np.float32)
            raw = decode_instances(region_probs, modality, body, core)
            features = instance_features(raw, region_probs)
            for f, params in enumerate(filters):
                scores[f, k] = pq_score(filter_instances(raw, features, **params), truth[sample_id][modality][0])[0]
        rows += [{"source": unet_source(scale), "scale": scale, "body": body, "core": core, "filter": f,
                  "pq": float(scores[f].mean())} for f in range(len(filters))]
    return rows


def cellpose_pq_table(flows: dict, truth: dict, modality: str, source: str) -> list[dict]:
    rows = []
    for cellprob, flow in CELLPOSE_GRID:
        config = {"source": source, "cellprob": cellprob, "flow": flow}
        scores = [pq_score(cellpose_labels(region_flows, modality, config), truth[sample_id][modality][0])[0]
                  for sample_id, region_flows in flows.items()]
        rows.append({**config, "pq": float(np.mean(scores))})
    return rows


def hybrid_pq_table(data: dict, truth: dict, modality: str, unet: str, cellpose: str) -> list[dict]:
    """Cellpose masks, kept or dropped by the U-Net's per-instance core mean and shape filters."""
    rows = []
    filters = FILTERS[modality]
    for cellprob, flow in CELLPOSE_GRID:
        config = {"source": "hybrid", "unet": unet, "cellpose": cellpose, "cellprob": cellprob, "flow": flow}
        scores = np.zeros((len(filters), len(data)))
        for k, (sample_id, (flows, unet_probs)) in enumerate(data.items()):
            labels = cellpose_labels(flows, modality, config)
            features = instance_features(labels, unet_probs.astype(np.float32))
            for f, params in enumerate(filters):
                scores[f, k] = pq_score(filter_instances(labels, features, **params), truth[sample_id][modality][0])[0]
        rows += [{**config, "filter": f, "pq": float(scores[f].mean())} for f in range(len(filters))]
    return rows


def pq_breakdown(labels: dict, truth: dict, modality: str) -> dict:
    """Pooled TP/FP/FN, mean IoU of true positives, and near misses (best IoU in 0.5-0.75)."""
    tp = fp = fn = near = 0
    iou_sum = 0.0
    for sample_id, pred in labels.items():
        gt = truth[sample_id][modality][0]
        npred, ngt = int(pred.max()), int(gt.max())
        joint = np.bincount((pred.astype(np.int64) * (ngt + 1) + gt).ravel(),
                            minlength=(npred + 1) * (ngt + 1)).reshape(npred + 1, ngt + 1)
        pc, gc = joint.sum(axis=1), joint.sum(axis=0)
        iou = joint[1:, 1:] / np.maximum(pc[1:, None] + gc[None, 1:] - joint[1:, 1:], 1)
        best = iou.max(axis=1) if ngt else np.zeros(npred)
        hits = best > 0.75
        tp += int(hits.sum())
        fp += npred - int(hits.sum())
        fn += ngt - int(hits.sum())
        near += int(((best > 0.5) & ~hits).sum())
        iou_sum += float(best[hits].sum())
    return {"tp": tp, "fp": fp, "fn": fn, "mean_tp_iou": iou_sum / max(tp, 1),
            "near_misses_iou_0.5_0.75": near}


def predicted_to_ground_truth(pred: np.ndarray, truth: np.ndarray) -> dict[int, int]:
    npred, ngt = int(pred.max()), int(truth.max())
    if not npred or not ngt:
        return {}
    joint = np.bincount((pred.astype(np.int64) * (ngt + 1) + truth).ravel(),
                        minlength=(npred + 1) * (ngt + 1)).reshape(npred + 1, ngt + 1)
    pc, gc = joint.sum(axis=1), joint.sum(axis=0)
    rows, cols = np.nonzero(joint[1:, 1:])
    rows += 1
    cols += 1
    iou = joint[rows, cols] / (pc[rows] + gc[cols] - joint[rows, cols])
    return {int(p): int(g) for p, g, score in zip(rows, cols, iou) if score > 0.75}


def ground_truth_ids(pred: np.ndarray, truth_labels: np.ndarray, ids: list[str]) -> list[str | None]:
    mapping = predicted_to_ground_truth(pred, truth_labels)
    return [ids[mapping[i] - 1] if i in mapping else None for i in range(1, int(pred.max()) + 1)]


def region_record(iv_labels: np.ndarray, ex_labels: np.ndarray, ex_image: np.ndarray, priors,
                  registrations=REGISTRATIONS) -> dict:
    iv_centers, ex_centers = region_centers(iv_labels), region_centers(ex_labels)
    n = int(ex_labels.max())
    counts = np.bincount(ex_labels.ravel(), minlength=n + 1)
    brightness = (np.bincount(ex_labels.ravel(), weights=normalize(ex_image, "exvivo")[0].ravel(),
                              minlength=n + 1) / np.maximum(counts, 1))[1:]
    circularity = instance_features(ex_labels)[:, 1]
    # "soma" weighting keeps dense bright processes from making a wrong alignment look good.
    target_weights = {"none": None, "soma": np.clip(brightness, 0, 1) * np.clip(circularity, 0, 1)}
    registration = {}
    for name in registrations:
        if name == "ncc":
            registration[name] = (register_ncc(iv_centers, ex_centers, priors),)
        else:
            registration[name] = tuple(register_with_priors(iv_centers, ex_centers, mode=mode,
                                                            target_weights=target_weights[name], priors=priors)
                                       for mode in (0, 1))
    return {"iv_centers": iv_centers, "ex_centers": ex_centers, "brightness": brightness,
            "iv_shape": iv_labels.shape, "ex_shape": ex_labels.shape, "registration": registration}


def subject_modes(records: dict, registration: str) -> dict:
    """Square ex-vivo canvases share one orientation per mouse; others choose per region."""
    modes = {}
    subjects = sorted({sample_id.split("__")[0] for sample_id in records})
    if registration == "ncc":
        return dict.fromkeys(subjects)
    for subject in subjects:
        items = [r for s, r in records.items() if s.startswith(subject)]
        square = np.mean([abs(np.log(r["ex_shape"][0] / r["ex_shape"][1])) < 0.1 for r in items])
        winners = np.array([r["registration"][registration][0][1] > r["registration"][registration][1][1]
                            for r in items])
        modes[subject] = None
        if square >= 0.65 and winners.mean() >= 0.65:
            modes[subject] = 0
        elif square >= 0.65 and (~winners).mean() >= 0.65:
            modes[subject] = 1
    return modes


def chosen_matrix(record: dict, registration: str, mode: int | None):
    candidates = record["registration"][registration]
    return (candidates[mode] if mode is not None else max(candidates, key=lambda c: c[1]))[0]


def consistent_regions(records: dict, matrices: dict, gate: float | None) -> set:
    """Regions whose in-vivo field lands near its mouse's median spot on the ex-vivo canvas.

    Within a mouse that spot varies by only a few percent of the canvas, so a far-off
    alignment is almost always wrong, and every pair it produces would be a false positive.
    """
    if gate is None:
        return set(records)
    # Anchor on the densest cluster of field centres (ties go to the training-mean position):
    # when a mouse's alignments split into two groups, a median lands between them and keeps none.
    centres = {}
    for sample_id, matrix in matrices.items():
        if matrix is not None:
            record = records[sample_id]
            centre = matrix[:, :2] @ np.array([record["iv_shape"][1] / 2, record["iv_shape"][0] / 2]) + matrix[:, 2]
            centres[sample_id] = centre / np.array([record["ex_shape"][1], record["ex_shape"][0]])
    keep = set()
    for subject in {sample_id.split("__")[0] for sample_id in centres}:
        members = [s for s in centres if s.startswith(subject)]
        points = np.array([centres[s] for s in members])
        near = np.linalg.norm(points[:, None] - points[None], axis=2) <= gate
        densest = np.flatnonzero(near.sum(axis=1) == near.sum(axis=1).max())
        seed = densest[np.argmin(np.linalg.norm(points[densest] - FIELD_PRIOR, axis=1))]
        anchor = np.median(points[near[seed]], axis=0)
        keep |= {s for s, point in zip(members, points) if np.linalg.norm(point - anchor) <= gate}
    return keep


def predict_pairs(record: dict, matrix, method: str, distance: float, min_brightness: float,
                  ratio: float | None = None):
    if matrix is None:
        return []
    keep = np.flatnonzero(record["brightness"] >= min_brightness)
    if len(keep) < 3:
        keep = np.arange(len(record["ex_centers"]))
    pairs = match_points(record["iv_centers"], record["ex_centers"][keep], matrix, distance, method)
    if ratio is None or not pairs:
        return [(i, int(keep[j])) for i, j, _ in pairs]
    # Ratio test: an unambiguous pair has no second ex-vivo candidate nearly as close.
    second = cKDTree(record["ex_centers"][keep]).query(transform(record["iv_centers"], matrix), k=2)[0][:, 1]
    return [(i, int(keep[j])) for i, j, d in pairs if second[i] >= ratio * max(d, 0.5)]


def cv_records(iv_probs: dict, ex_probs: dict, truth: dict, iv_config: dict, ex_config: dict,
               priors_by_subject: dict) -> dict:
    records = {}
    for sample_id, item in truth.items():
        iv = decode(iv_probs[sample_id], "invivo", iv_config)
        ex = decode(ex_probs[sample_id], "exvivo", ex_config)
        ex_image = read_image(region_path(sample_id) / "exvivo.tif")
        record = region_record(iv, ex, ex_image, priors_by_subject[item["subject"]])
        record["iv_gt"] = ground_truth_ids(iv, *item["invivo"])
        record["ex_gt"] = ground_truth_ids(ex, *item["exvivo"])
        records[sample_id] = record
    return records


def matching_search(records: dict, truth: dict) -> list[dict]:
    total = sum(len(item["pairs"]) for item in truth.values())
    results = []
    for registration in REGISTRATIONS:
        modes = subject_modes(records, registration)
        matrices = {s: chosen_matrix(r, registration, modes[s.split("__")[0]]) for s, r in records.items()}
        kept = {gate: consistent_regions(records, matrices, gate) for gate in CENTRE_GATES}
        for method, distance, level, ratio in itertools.product(METHODS, DISTANCES, BRIGHTNESS_LEVELS, RATIOS):
            counts = {}
            for sample_id, record in records.items():
                pairs = predict_pairs(record, matrices[sample_id], method, distance, level, ratio)
                counts[sample_id] = (sum((record["iv_gt"][i], record["ex_gt"][j]) in truth[sample_id]["pairs"]
                                         for i, j in pairs), len(pairs))
            for gate, regions in kept.items():
                tp = sum(counts[s][0] for s in regions)
                predicted = sum(counts[s][1] for s in regions)
                results.append({"registration": registration, "centre_gate": gate, "method": method,
                                "distance": distance, "min_brightness": level, "ratio": ratio,
                                "tp": tp, "predicted": predicted,
                                "precision": tp / max(predicted, 1), "recall": tp / total,
                                "f1": 2 * tp / (predicted + total)})
    return results


def tune(caches: dict, truth: dict, top: int = 2) -> dict:
    """caches[(modality, source)] -> {sample_id: probs or flows}; adds hybrid sources in place.

    Returns the best leak-free config. Matching is searched for the best setting of each of the
    top `top` segmentation sources per modality.
    """
    segmentation = {m: [] for m in MODALITIES}
    for (modality, source), data in list(caches.items()):
        started = time.monotonic()
        if source.startswith("cellpose"):
            segmentation[modality] += cellpose_pq_table(data, truth, modality, source)
        else:
            segmentation[modality] += pq_table(data, truth, modality, float(source.removeprefix("unet_x")))
        print(f"PQ grid {modality} {source}: {time.monotonic() - started:.0f}s", flush=True)
    for modality in MODALITIES:
        unets = [r for r in segmentation[modality] if r["source"].startswith("unet")]
        cellposes = [r for r in segmentation[modality] if r["source"].startswith("cellpose")]
        if cellposes and unets:
            unet = max(unets, key=lambda row: row["pq"])["source"]
            cellpose = max(cellposes, key=lambda row: row["pq"])["source"]
            started = time.monotonic()
            caches[(modality, "hybrid")] = {s: (flows, caches[(modality, unet)][s])
                                            for s, flows in caches[(modality, cellpose)].items()}
            segmentation[modality] += hybrid_pq_table(caches[(modality, "hybrid")], truth, modality, unet, cellpose)
            print(f"PQ grid {modality} hybrid ({cellpose} + {unet}): {time.monotonic() - started:.0f}s", flush=True)
    for modality in MODALITIES:
        segmentation[modality].sort(key=lambda row: row["pq"], reverse=True)
        print(f"\nTop {modality} settings (held-out PQ):")
        for row in segmentation[modality][:5]:
            print("  ", row, FILTERS[modality][row["filter"]] if "filter" in row else "")
        for source in sorted({row["source"] for row in segmentation[modality]}):
            print(f"   best {source}: {max(r['pq'] for r in segmentation[modality] if r['source'] == source):.4f}")
    priors = {subject: estimate_linear_priors({subject}) for subject in SUBJECTS}
    total = sum(len(item["pairs"]) for item in truth.values())
    best = None
    candidates = {}
    for modality in MODALITIES:
        per_source = {}
        for row in segmentation[modality]:
            per_source.setdefault(row["source"], row)
        candidates[modality] = sorted(per_source.values(), key=lambda row: row["pq"], reverse=True)[:top]
    for iv_config, ex_config in itertools.product(candidates["invivo"], candidates["exvivo"]):
        records = cv_records(caches[("invivo", iv_config["source"])], caches[("exvivo", ex_config["source"])],
                             truth, iv_config, ex_config, priors)
        eligible = sum(a in set(r["iv_gt"]) and b in set(r["ex_gt"])
                       for s, r in records.items() for a, b in truth[s]["pairs"])
        segmentation_score = 0.25 * (iv_config["pq"] + ex_config["pq"])
        match = max(matching_search(records, truth), key=lambda row: row["f1"])
        final = segmentation_score + 0.5 * match["f1"]
        print(f"\nIV {iv_config} | EX {ex_config}\n  eligible pairs {eligible}/{total} "
              f"(F1 ceiling {2 * eligible / (eligible + total):.3f}) best match {match}\n"
              f"  CV score = {final:.4f}", flush=True)
        if best is None or final > best["cv_score"]:
            best = {"cv_score": final, "invivo": iv_config, "exvivo": ex_config, "match": match}
    for modality in MODALITIES:
        config = best[modality]
        labels = {s: decode(caches[(modality, config["source"])][s], modality, config) for s in truth}
        print(f"\n{modality} PQ breakdown for the chosen setting:", pq_breakdown(labels, truth, modality))
    return best


def assemble_submission(predictions: dict, best: dict, output: Path) -> Path:
    """predictions[sample_id][modality] is what decode() needs for best[modality] on that test region."""
    sample = pd.read_csv(ROOT / "sample_submission.csv")
    priors = estimate_linear_priors()
    match = best["match"]
    records, labels = {}, {}
    for sample_id in sample.sample_id:
        region_labels = {modality: decode(predictions[sample_id][modality], modality, best[modality])
                         for modality in MODALITIES}
        ex_image = read_image(region_path(sample_id, "hidden_test") / "exvivo.tif")
        records[sample_id] = region_record(region_labels["invivo"], region_labels["exvivo"], ex_image,
                                           priors, registrations=(match["registration"],))
        labels[sample_id] = region_labels
        print("decoded", sample_id, {m: int(l.max()) for m, l in region_labels.items()}, flush=True)
    modes = subject_modes(records, match["registration"])
    matrices = {s: chosen_matrix(r, match["registration"], modes[s.split("__")[0]]) for s, r in records.items()}
    kept = consistent_regions(records, matrices, match["centre_gate"])
    print(f"centre gate {match['centre_gate']}: pairs kept for {len(kept)}/{len(records)} regions", flush=True)
    rows = []
    for sample_id, record in records.items():
        pairs = (predict_pairs(record, matrices[sample_id], match["method"], match["distance"],
                               match["min_brightness"], match["ratio"])
                 if sample_id in kept else [])
        rows.append({"sample_id": sample_id,
                     "invivo_instances": labels_to_rles(labels[sample_id]["invivo"], "IVP"),
                     "exvivo_instances": labels_to_rles(labels[sample_id]["exvivo"], "EXP"),
                     "match_pairs": [[f"IVP_{i + 1:06d}", f"EXP_{j + 1:06d}"] for i, j in pairs]})
    write_submission(rows, output)
    print(f"saved {output} with {sum(len(r['match_pairs']) for r in rows)} pairs", flush=True)
    return output


def make_submission(models: dict, best: dict, output: Path, tta: bool = True) -> Path:
    """models[modality] = {"cellpose": [Cellpose models] or None, "unet": [U-Nets] or None}.

    Several Cellpose models (final + fold fine-tunes) are averaged at the flow level.
    """
    sample = pd.read_csv(ROOT / "sample_submission.csv")
    predictions = {}
    for sample_id in sample.sample_id:
        path = region_path(sample_id, "hidden_test")
        predictions[sample_id] = {}
        for modality in MODALITIES:
            image = read_image(path / f"{modality}.tif")
            config = best[modality]
            if config["source"].startswith("cellpose"):
                prediction = cellpose_flows(models[modality]["cellpose"], image)
            elif config["source"] == "hybrid":
                scale = float(config["unet"].removeprefix("unet_x"))
                prediction = (cellpose_flows(models[modality]["cellpose"], image),
                              np.mean([predict_probabilities(model, image, modality, DEVICE, tta=tta, scale=scale)
                                       for model in models[modality]["unet"]], axis=0))
            else:
                prediction = np.mean([predict_probabilities(model, image, modality, DEVICE, tta=tta,
                                                            scale=config["scale"])
                                      for model in models[modality]["unet"]], axis=0)
            predictions[sample_id][modality] = prediction
        print("inferred", sample_id, flush=True)
    return assemble_submission(predictions, best, output)


In [ ]:
%%writefile validate_submission.py
"""Strict structural checks for the generated Kaggle submission."""

from __future__ import annotations

import argparse
import csv
import json
import sys
from pathlib import Path

import numpy as np

from cellmatch import ROOT, read_image


def check_instances(instances: dict[str, str], shape: tuple[int, int]) -> int:
    occupied = np.zeros(shape[0] * shape[1], np.bool_)
    total = 0
    for cell_id, rle in instances.items():
        if not isinstance(cell_id, str) or not cell_id:
            raise ValueError("Invalid cell ID")
        runs = np.fromstring(rle, sep=" ", dtype=np.int64)
        if len(runs) == 0 or len(runs) % 2:
            raise ValueError(f"Invalid run count: {cell_id}")
        for start, length in runs.reshape(-1, 2):
            if start < 0 or length <= 0 or start + length > occupied.size:
                raise ValueError(f"RLE out of bounds: {cell_id}")
            if occupied[start:start + length].any():
                raise ValueError(f"Overlapping masks: {cell_id}")
            occupied[start:start + length] = True
            total += int(length)
    return total


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("path", type=Path)
    args = parser.parse_args()
    # A region's RLE field can exceed the csv module's default 128 KB limit.
    csv.field_size_limit(sys.maxsize)
    with (ROOT / "sample_submission.csv").open(newline="") as stream:
        expected = [row["sample_id"] for row in csv.DictReader(stream)]
    with args.path.open(newline="") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == ["sample_id", "invivo_instances", "exvivo_instances", "match_pairs"]
        rows = list(reader)
    if [row["sample_id"] for row in rows] != expected:
        raise ValueError("Rows do not match sample_submission.csv")
    total = {"invivo_cells": 0, "exvivo_cells": 0, "matches": 0}
    for row in rows:
        iv, ex, pairs = (json.loads(row[key]) for key in
                         ("invivo_instances", "exvivo_instances", "match_pairs"))
        if not isinstance(iv, dict) or not isinstance(ex, dict) or not isinstance(pairs, list):
            raise ValueError(f"Wrong JSON type in {row['sample_id']}")
        path = ROOT / "hidden_test" / row["sample_id"].replace("__", "/")
        for modality, instances in (("invivo", iv), ("exvivo", ex)):
            shape = read_image(path / f"{modality}.tif").shape
            check_instances(instances, shape)
            total[f"{modality}_cells"] += len(instances)
        if len({a for a, _ in pairs}) != len(pairs) or len({b for _, b in pairs}) != len(pairs):
            raise ValueError(f"Repeated match IDs in {row['sample_id']}")
        if any(a not in iv or b not in ex for a, b in pairs):
            raise ValueError(f"Match refers to absent mask in {row['sample_id']}")
        total["matches"] += len(pairs)
    print(f"PASS {len(rows)} rows, {total}, {args.path.stat().st_size:,} bytes")


if __name__ == "__main__":
    main()


## Config

In [ ]:
import json, time
import torch
from pathlib import Path
import pipeline as P

WIDTH = 32                                   # U-Net base channels
STEPS = {"invivo": 3000, "exvivo": 4000}     # U-Net optimizer steps per model
BATCH = 16
SCALES = (1.0,)                              # U-Net input scales; x2 lost in both modalities before
FINAL_SEEDS = (0, 1)                         # all-data U-Net ensemble members per modality
TTA = True

# Cellpose-SAM fine-tuning budgets, name -> (epochs, cell-centred tiles per epoch).
# "cellpose" is the budget already trained and stored on Drive; "cellpose_long" is new.
CELLPOSE_VARIANTS = {"cellpose": (60, 64), "cellpose_long": (150, 96)}
CELLPOSE_BATCH = 4
print(P.DEVICE, P.ROOT, "| bf16:", torch.cuda.is_bf16_supported())

## Stage 1 — leave-one-mouse-out training

In [ ]:
started = time.time()
for scale in SCALES:
    for subject in P.SUBJECTS:
        others = [s for s in P.SUBJECTS if s != subject]
        for modality in P.MODALITIES:
            P.train(modality, others, P.model_path(modality, WIDTH, scale, f"fold_{subject}", 0),
                    STEPS[modality], WIDTH, scale, seed=0, batch=BATCH)
print(f"U-Net fold training: {(time.time() - started) / 60:.1f} min")

In [ ]:
started = time.time()
for variant, (epochs, per_epoch) in CELLPOSE_VARIANTS.items():
    suffix = variant.removeprefix("cellpose")
    for subject in P.SUBJECTS:
        others = [s for s in P.SUBJECTS if s != subject]
        for modality in P.MODALITIES:
            P.train_cellpose(modality, others, P.cellpose_path(modality, f"fold_{subject}{suffix}"),
                             epochs, per_epoch, CELLPOSE_BATCH)
print(f"Cellpose-SAM fold fine-tuning: {(time.time() - started) / 60:.1f} min")

## Stage 2 — held-out predictions, then leak-free search on the competition score

In [ ]:
started = time.time()
truth = P.load_truth()
caches = {(modality, P.unet_source(scale)): P.fold_probabilities(truth, modality, WIDTH, scale, tta=TTA)
          for scale in SCALES for modality in P.MODALITIES}
print(f"U-Net held-out inference: {(time.time() - started) / 60:.1f} min")
for variant in CELLPOSE_VARIANTS:
    started = time.time()
    for modality in P.MODALITIES:
        caches[(modality, variant)] = P.fold_cellpose_flows(truth, modality, variant)
    print(f"{variant} held-out inference: {(time.time() - started) / 60:.1f} min")

In [ ]:
started = time.time()
best = P.tune(caches, truth)
Path("best_config.json").write_text(json.dumps(best, indent=2))
(MODELS / "best_config.json").write_text(json.dumps(best, indent=2))
print(f"\nsearch: {(time.time() - started) / 60:.1f} min")
print("CHOSEN (leave-one-mouse-out CV score %.4f):" % best["cv_score"])
print(json.dumps(best, indent=2))


## Stage 3 — final models on all mice, test predictions, `submission.csv`

In [ ]:
del caches
started = time.time()
models = {}
for modality in P.MODALITIES:
    config = best[modality]
    entry = {"cellpose": None, "unet": None}
    variant = config.get("cellpose", config["source"])
    if variant.startswith("cellpose"):
        epochs, per_epoch = CELLPOSE_VARIANTS[variant]
        suffix = variant.removeprefix("cellpose")
        final = P.train_cellpose(modality, P.SUBJECTS, P.cellpose_path(modality, f"all{suffix}"),
                                 epochs, per_epoch, CELLPOSE_BATCH)
        paths = [final] + [P.cellpose_path(modality, f"fold_{s}{suffix}") for s in P.SUBJECTS]
        entry["cellpose"] = [P.cellpose_model(path) for path in paths]
    if not config["source"].startswith("cellpose"):
        scale = config.get("scale") or float(config["unet"].removeprefix("unet_x"))
        paths = [P.train(modality, P.SUBJECTS, P.model_path(modality, WIDTH, scale, "all", seed),
                         STEPS[modality], WIDTH, scale, seed=seed, batch=BATCH)
                 for seed in FINAL_SEEDS]
        entry["unet"] = [P.load_model(path, P.DEVICE) for path in paths]
    models[modality] = entry
    print(modality, "->", config["source"], "| cellpose models:", len(entry["cellpose"] or []),
          "| unets:", len(entry["unet"] or []))
print(f"final training: {(time.time() - started) / 60:.1f} min")

In [ ]:
submission = P.make_submission(models, best, WORK / "submission.csv", tta=TTA)
(MODELS / "submission.csv").write_text((WORK / "submission.csv").read_text())
get_ipython().system(f"python validate_submission.py {WORK / 'submission.csv'}")
print("DONE →", WORK / "submission.csv")
if ON_COLAB:
    from google.colab import files
    files.download(str(WORK / "submission.csv"))
